# Computer Vision Training of Digits with Three Different Models

## 1. Load data

Python Environment 3.12.12

In [1]:
%pip install --upgrade pip
%pip install torch torchvision matplotlib pandas streamlit pillow

Note: you may need to restart the kernel to use updated packages.
ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Compose steps: step1, pixel to tensor; step 2, normalize
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# Load  Train data
train_full = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

# Load test data
test_data = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_data, val_data = random_split(
    train_full,
    [50000, 10000],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(
    train_data, batch_size=128, shuffle=True
)
val_loader = DataLoader(
    val_data, batch_size=128
)
test_loader = DataLoader(
    test_data, batch_size=128
)

## 2. Verify the shape

Other data clean steps are ommitted for this version

In [ ]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

## 3. Prepare models
### Model 1 - MLP

In [ ]:
import torch.nn as nn

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

### Model 2 - CNN

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 7x7

            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.net(x)

### Model 3 - Transformer

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()

        dim = 64

        # 28x28 -> 4x4 patches -> 16 tokens
        self.patch_embed = nn.Conv2d(
            1, dim, kernel_size=7, stride=7
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(1, 16, dim)
        )
        nn.init.normal_(self.pos_embed, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=4,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True,
            norm_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
            enable_nested_tensor=False
        )

        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, 10)

    def forward(self, x):
        x = self.patch_embed(x)   # [B, 64, 4, 4]
        x = x.flatten(2)         # [B, 64, 16]
        x = x.transpose(1, 2)    # [B, 16, 64]

        x = x + self.pos_embed
        x = self.encoder(x)

        # Mean pooling over patch tokens
        x = self.norm(x.mean(dim=1))

        return self.head(x)

## 4. Train models

In [ ]:
import time
import torch
import pandas as pd

device = torch.device(
    # cuda environment locally
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    loss_sum = 0.0
    criterion = nn.CrossEntropyLoss()

    with torch.inference_mode():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)
            loss = criterion(logits, labels)

            loss_sum += loss.item() * labels.size(0)
            correct += (
                logits.argmax(dim=1) == labels
            ).sum().item()
            total += labels.size(0)

    return {
        "loss": loss_sum / total,
        "accuracy": correct / total
    }


def train(model, epochs=5):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=0.001
    )
    criterion = nn.CrossEntropyLoss()

    start = time.perf_counter()
    best_val_accuracy = -1

    for epoch in range(epochs):
        model.train()

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images)

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        val = evaluate(model, val_loader)

        # Printout evaluation result
        print(
            f"Epoch {epoch + 1}: "
            f"validation accuracy = {val['accuracy']:.4f}"
        )

        if val["accuracy"] > best_val_accuracy:
            best_val_accuracy = val["accuracy"]
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

    # Restore best validation checkpoint
    model.load_state_dict(best_state)
    model.eval()

    return model, time.perf_counter() - start

## 5. Test Models

Retrieve the result with three different models

In [ ]:
torch.manual_seed(42)

models = {
    "MLP": MLP(),
    "CNN": CNN(),
    "Transformer": VisionTransformer()
}

results = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}")

    trained, elapsed = train(model, epochs=5)
    metrics = evaluate(trained, test_loader)

    parameters = sum(
        p.numel() for p in trained.parameters()
    )

    torch.save(
        trained.cpu().state_dict(),
        f"{name.lower()}.pt"
    )
    trained.to(device)

    trained_models[name] = trained

    results.append({
        "model": name,
        "test_accuracy": metrics["accuracy"],
        "test_loss": metrics["loss"],
        "training_seconds": elapsed,
        "parameters": parameters
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))